# Splitting and data leakage

**Accompanies Section 5 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

Random splitting is not a defensible default for molecular data. Leakage is any route by which information about your test set reaches your model before you evaluate it, and the usual route in molecular work is a test molecule that is a near-duplicate of a training molecule, which makes the model look better than it is. This notebook *measures* how much near-duplicate leakage each splitting strategy admits, so a claim about methodology becomes a number you can report.

### Learning objectives
- Build random, scaffold, and cluster splits of the same data set
- Quantify near-duplicate leakage across each
- See why a scaffold split degenerates on large, modular molecules
- Demonstrate preprocessing leakage directly, by fitting a scaler on the wrong data
- Understand why unsupervised pipelines still need splits

### What this notebook is designed to make go wrong
A random split that looks reasonable and leaves near-identical molecules on both sides, a scaffold split that barely does better because the scaffold is nearly the whole molecule, and a scaler fitted before splitting that sees the test set.

### What you need installed
RDKit, scikit-learn, pandas, NumPy and matplotlib. The `environment.yml` at the repository root installs them; nothing optional.

### Roughly how long it takes
A couple of minutes. The leakage measurement compares every test molecule against every training molecule.

In [ ]:
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. Load and standardize the molecules

The choice of data set is the whole demonstration here. A random sample of ZINC-250k is drug-like
and diverse, so its molecules already sit far apart in fingerprint space and *no* split strands
near-duplicates, for the simple reason that there are none to strand.

We use the PROTAC set instead. Targeted protein degraders are published in analogue families that
differ only in linker length or a single stereocenter, so the set is dense with near-duplicates,
which is exactly the redundancy the article (Section 5.2) warns a random split will leak. The same
molecules are large and modular, so the Bemis-Murcko scaffold absorbs nearly the whole structure
and the scaffold split barely helps. Both effects are measured below.

Read the CSV and put every structure into one convention first. Notebook 04 walks through the
standardization pipeline step by step; the short version below keeps four of those steps: parse,
keep the largest fragment, neutralize the charge, write a canonical SMILES. Then deduplicate,
because a leakage measurement on a set that still holds the same molecule twice measures your
bookkeeping instead of your split.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")   # parse failures return None; they are not printed

LARGEST_FRAGMENT = rdMolStandardize.LargestFragmentChooser()
UNCHARGER = rdMolStandardize.Uncharger()


def standardize(smiles):
    """Parse, strip salts, neutralize, canonicalize. None if the string is unusable."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    if "." in smiles:
        mol = LARGEST_FRAGMENT.choose(mol)
    return Chem.MolToSmiles(UNCHARGER.uncharge(mol))


# The PROTAC set: 1,000 targeted protein degraders, published in analogue
# families, which is what gives a random split something to leak. Stereochemistry
# is kept, because some of these families differ only by a stereocenter and
# collapsing them before the split would hide how each split treats them.
protac = pd.read_csv(DATA / "protac-tpddb-sample.csv")
smiles = sorted({s for s in (standardize(x) for x in protac["smiles"].str.strip())
                 if s is not None})
print(f"{len(protac)} records in, {len(smiles)} unique standardized molecules out")

### Look at what you are clustering

These are targeted protein degraders, and they do not look like ordinary drug-like molecules: each is a warhead and an E3-ligand joined by a linker, two to three times the size of a screening compound and visibly modular. That structure is the reason the scaffold split below behaves the way it does, so it is worth seeing before the numbers arrive.

In [ ]:
from rdkit.Chem import Draw
from IPython.display import display


def draw_molecules(smiles_list, legends=None, n=8, per_row=4):
    """Grid depiction of the first n molecules, for orientation rather than analysis.

    Molecules that fail to parse are dropped along with their legend, so a single
    bad string does not blank the whole grid.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list[:n]]
    legs = list(legends[:n]) if legends is not None else None
    keep = [i for i, m in enumerate(mols) if m is not None]
    mols = [mols[i] for i in keep]
    legs = [legs[i] for i in keep] if legs is not None else None
    return Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=(260, 200), legends=legs)


# Eight PROTACs, labeled by the E3 ligase they recruit and the target they
# degrade, so the reader meets the objects before the analysis treats them as points.
protac_legends = [f"{lig} / {tgt}" for lig, tgt in zip(protac["ligase"], protac["target_symbol"])]
display(draw_molecules(protac["smiles"].tolist(), legends=protac_legends, n=8))

## 2. Build three splits of the same data

We build a random split, a Bemis-Murcko scaffold split, and a similarity-cluster split. All
three hold out roughly 20% of the molecules, and they differ only in *which* 20%.

Both chemistry-aware splits work the same way. Give every molecule a group label, then send
whole groups to one side of the partition so that no group straddles it. The scaffold split
labels each molecule by its Bemis-Murcko scaffold: strip every side chain, keep the ring
systems and the chains that connect them. Holding whole scaffolds out is stricter than a
random split, and it is the standard way to claim a model generalizes to new chemistry. The
cluster split labels each molecule by a Taylor-Butina cluster over Morgan fingerprints,
which is stricter still, because two molecules can share no scaffold and still be
near-duplicates.

Those clusters run on Tanimoto similarity: for two binary fingerprints, the number of bits
both have set divided by the number either has set, from 0 to 1. Jaccard distance is 1 minus
that, and it is what the clustering below consumes. The scale does not carry across
fingerprint types, since 0.7 between two ECFP4 fingerprints and 0.7 between two MACCS keys
mean different things, and it also tracks molecular size, so a tight cluster can turn out to
be a set of similarly sized molecules.

The group assignment below visits the largest group first and takes it only when doing so
moves the test set closer to the target size. The obvious version, adding groups while the
test set sits below target, overshoots badly when one series dominates: a single scaffold
covering 60% of a library turns a request for 20% into a 60% test set, and congeneric series
do dominate real chemical libraries. Group sizes can also make the fraction you asked for
unreachable, so print the fraction you achieved and report that one.

In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Cluster import Butina

MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)


def bemis_murcko_scaffolds(smiles_list):
    """The ring systems plus connecting linkers of each molecule, as a SMILES string."""
    groups = []
    for i, smi in enumerate(smiles_list):
        core = MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(smi))
        key = Chem.MolToSmiles(core)
        # An acyclic molecule has an empty scaffold. Give each one its own group;
        # pooling them would create one group that straddles the split.
        groups.append(key if key else f"acyclic_{i}")
    return np.array(groups)


def butina_clusters(smiles_list, tanimoto_cutoff=0.65):
    """Cluster index per molecule, from Taylor-Butina clustering on Morgan fingerprints.

    Butina.ClusterData takes a DISTANCE threshold, not a similarity, so a Tanimoto
    cutoff of 0.65 (molecules at least that similar cluster together) is passed as
    the distance 1 - 0.65 = 0.35. Passing 0.65 directly, as an earlier version did,
    silently clustered at Tanimoto 0.35 instead.
    """
    fps = [MORGAN.GetFingerprint(Chem.MolFromSmiles(s)) for s in smiles_list]
    # Butina wants the lower triangle of the distance matrix, flattened.
    distances = []
    for i in range(1, len(fps)):
        distances.extend(1.0 - s for s in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i]))
    clusters = Butina.ClusterData(distances, len(fps), 1.0 - tanimoto_cutoff, isDistData=True)

    labels = np.empty(len(fps), dtype=int)
    for cluster_id, members in enumerate(clusters):
        labels[list(members)] = cluster_id
    return labels


def group_split(groups, test_size=0.2, random_state=SEED):
    """Hold out whole groups, landing as close to `test_size` as the groups allow."""
    jitter = np.random.default_rng(random_state)
    unique, counts = np.unique(groups, return_counts=True)
    # Largest group first, with a small jitter so that ties break reproducibly.
    order = np.argsort(-counts + jitter.uniform(0, 0.5, size=counts.shape))
    target = test_size * len(groups)

    held_out, n_held = [], 0
    for idx in order:
        if abs(n_held + counts[idx] - target) < abs(n_held - target):
            held_out.append(unique[idx])
            n_held += counts[idx]

    mask = np.isin(groups, held_out)
    return np.where(~mask)[0], np.where(mask)[0]

In [ ]:
n = len(smiles)
perm = np.random.default_rng(SEED).permutation(n)
n_test = int(0.2 * n)
random_split = (perm[n_test:], perm[:n_test])

scaffold = group_split(bemis_murcko_scaffolds(smiles), test_size=0.2)
cluster = group_split(butina_clusters(smiles, tanimoto_cutoff=0.65), test_size=0.2)

splits = [("random", random_split), ("scaffold", scaffold), ("Butina cluster", cluster)]
for name, (train, test) in splits:
    print(f"{name:10s} train {len(train):4d}  test {len(test):4d}   "
          f"{len(test) / n:.1%} held out")

> **Watch the scaffold split barely help below.** Bemis-Murcko was defined on drug-sized
> molecules, where the core summarizes the molecule. These are PROTACs, where the warhead, the
> linker and the E3 ligand are all ring systems joined by chains, so the scaffold absorbs nearly
> every heavy atom and almost every compound gets its own scaffold. The split then degenerates
> into a random one with extra machinery, and the near-duplicate rate it admits (measured next)
> sits close to the random split's rather than well below it. Notebook 08, Section 7.6 quantifies
> the scaffold-coverage diagnostic; on a set like this, split on the shared warhead, the E3 ligand
> or the linker class instead.

### See the families the split has to respect

The reason a random split leaks here is that the library is built from families, and a picture
makes that concrete. The map below is a t-SNE of the Morgan fingerprints on Jaccard distance,
which is a visualization rather than evidence (Section 7 explains why), so read it as a suggestion
of where the families sit. Colored by the E3 ligase, the set divides into two broad territories,
because every CRBN degrader shares a glutarimide and every VHL degrader shares the VHL ligand.
Colored by the degradation target, tighter islands appear inside those territories, since
molecules that degrade the same target tend to be analogues of one another. A random split
scatters the members of an island across train and test, which is the near-duplicate leakage
measured next; the Taylor-Butina cluster split keeps each island whole, and the scaffold split,
for the reasons above, mostly does not.

In [ ]:
from sklearn.manifold import TSNE

# Rebuild an aligned table: standardized SMILES with the ligase and target of the
# first record that produced each unique structure, so the point colors line up
# with the molecules.
aligned = {}
for smi, lig, tgt in zip(protac["smiles"].str.strip(), protac["ligase"], protac["target_symbol"]):
    std = standardize(smi)
    if std is not None and std not in aligned:
        aligned[std] = (lig, tgt)
keys = list(aligned)
ligase = np.array([aligned[k][0] for k in keys])
target = np.array([aligned[k][1] for k in keys])
fps = np.array([MORGAN.GetFingerprintAsNumPy(Chem.MolFromSmiles(k)) for k in keys]).astype(bool)

# A 2D map for orientation only (Section 7): t-SNE on Jaccard distances between
# fingerprints. Read it as a suggestion of where the families sit, not as proof
# the islands are real.
emb = TSNE(n_components=2, metric="jaccard", init="random",
           perplexity=30, random_state=SEED).fit_transform(fps)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

for lig, color in zip(["CRBN", "VHL"], [TEAL, PLUM]):
    m = ligase == lig
    axes[0].scatter(emb[m, 0], emb[m, 1], s=9, c=color, alpha=0.6, linewidths=0, label=lig)
axes[0].legend(title="E3 ligase", frameon=False)
axes[0].set_title("Colored by E3 ligase")

top = pd.Series(target).value_counts().index[:6].tolist()
axes[1].scatter(emb[~np.isin(target, top), 0], emb[~np.isin(target, top), 1],
                s=9, c="#CCCCCC", alpha=0.5, linewidths=0, label="other")
for tgt, color in zip(top, [PURPLE, GREEN, LAVENDER, PLUM, TEAL, SLATE]):
    m = target == tgt
    axes[1].scatter(emb[m, 0], emb[m, 1], s=9, c=color, alpha=0.75, linewidths=0, label=tgt)
axes[1].legend(title="target", frameon=False, fontsize=7)
axes[1].set_title("Colored by degradation target")

for ax in axes:
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()

## 3. Measure the leakage

For each test molecule, find its most similar training molecule. If that similarity is high,
the model or embedding is being evaluated on something it has already seen.

**This is the number to report alongside any split.** It converts "we used a scaffold split"
from an assertion into a measurement.

In [ ]:
THRESHOLD = 0.7


def nearest_training_similarity(smiles_list, train_idx, test_idx):
    """For each test molecule, the Tanimoto similarity to its closest training molecule."""
    train_fps = [MORGAN.GetFingerprint(Chem.MolFromSmiles(smiles_list[i])) for i in train_idx]
    test_fps = [MORGAN.GetFingerprint(Chem.MolFromSmiles(smiles_list[i])) for i in test_idx]
    return np.array([max(DataStructs.BulkTanimotoSimilarity(fp, train_fps)) for fp in test_fps])


nearest = {}
for name, (train, test) in splits:
    sims = nearest_training_similarity(smiles, train, test)
    nearest[name] = sims
    leaking = sims >= THRESHOLD
    print(f"{name:10s} train {len(train)} / test {len(test)}; "
          f"{leaking.sum()} test molecules ({100 * leaking.mean():.1f}%) have a training "
          f"neighbor above Tanimoto {THRESHOLD}; median nearest-neighbor similarity "
          f"{np.median(sims):.3f}, max {sims.max():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))
names = list(nearest)
fractions = [100 * (nearest[k] >= THRESHOLD).mean() for k in names]
medians = [np.median(nearest[k]) for k in names]

bars = ax.bar(names, fractions, color=PALETTE[:3])
for bar, med in zip(bars, medians):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"median NN\nsimilarity\n{med:.2f}", ha="center", fontsize=7)
ax.set_ylabel("% of test molecules with a\ntraining neighbor above Tanimoto 0.7")
ax.set_title("Stricter splits admit less near-duplicate leakage")
ax.set_ylim(0, max(fractions) * 1.45 + 1)
plt.show()

## 4. Fit a scaler the wrong way and watch it leak

Preprocessing leakage is the most common leak in practice and the easiest to fix. Below, a
`StandardScaler` is fitted two ways: on the full data set (wrong) and on the training portion
only (right), and the difference in the resulting test-set statistics is the information that
leaked.

In [ ]:
from sklearn.preprocessing import StandardScaler
from rdkit.Chem.Descriptors import MolLogP, MolWt, NumHAcceptors, NumHDonors, TPSA

def descriptors(smiles_list):
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        rows.append([MolWt(mol), MolLogP(mol), TPSA(mol), NumHDonors(mol), NumHAcceptors(mol)])
    return np.array(rows)

X = descriptors(smiles)
train_idx, test_idx = scaffold

wrong = StandardScaler().fit(X)                 # LEAK: has seen the test set
right = StandardScaler().fit(X[train_idx])      # correct: training data only

X_test_wrong = wrong.transform(X[test_idx])
X_test_right = right.transform(X[test_idx])

print("Test-set mean of each standardized descriptor:")
print(f"  scaler fitted on ALL data   : {np.round(X_test_wrong.mean(axis=0), 3)}")
print(f"  scaler fitted on TRAIN only : {np.round(X_test_right.mean(axis=0), 3)}")
print("\nThe leaky scaler pulls every test-set mean toward zero, because it was fitted")
print("on data that already included the test set. The train-only scaler leaves them")
print("further out, and that offset is real information about generalization that the")
print("leaky version threw away.")

### Exercise

The cluster split above uses a Tanimoto cutoff of 0.65, meaning molecules at least that similar
are placed in one cluster (`butina_clusters` converts it to the distance threshold Butina
actually takes, `1 - 0.65`). Vary `tanimoto_cutoff` from 0.4 to 0.9, where a higher value is
stricter and makes smaller, tighter clusters, and plot the near-duplicate leakage against it.
What value would you choose, and what does the trade-off against test-set size look like?

In [ ]:
# YOUR CODE HERE